# slab_two_pistons — converter to the two-piston (feed / permeate) geometry

Turns an **equilibrated** `slab_with_support` periodic snapshot into the input for
`triaxial_permeation_two_pist.lmp` / `triaxial_compression_two_pist.lmp` (2026-09-16):

```
zlo = 0
| vacuum margin (permeate side)
| permeate piston (type 6)   ← new
| permeate reservoir = existing below-support solvent, PADDED to permeate_thickness
| support sheet (type 4, frozen, unchanged)
| gap | gel slab | feed reservoir (existing top solvent)
|   [dry piston, type 7, parked mid-feed]   ← new
| feed piston (type 5)       ← new
| vacuum margin (feed side)
zhi
```

**Why a converter, not a fresh lattice.**  The NPT-piston scheme (Marioni et al., J. Membr. Sci. 738 (2026)
124837, Eq. 3) controls pressure in *z only* — `lx`, `ly` stay fixed for the whole run — so it can never swell a
fresh `pre_swell = 0.93` lattice laterally.  Starting from the `slab_with_support` final state (aniso NPH at P*,
piston transparent to solvent, σ_p,xx/σ_p,zz = 1.0005) the gel arrives with zero transverse network stress and its
equilibrium `lx`, `ly` and gel dimensions *by construction*; the NPT-piston phase in the new decks is then only a
z-settle of the reservoirs.  The old type-5 piston is deleted; gel and solvent coordinates are copied verbatim (one
uniform z shift).

All code lives in `scripts/slab_two_pistons.py` (importable module + CLI); this notebook only sets the Config,
runs it, and shows the validation output and the z-density histogram.

> **Input file.**  The production input is the 2026-09-07 `PISTON_TRANSPARENT=1` rerun
> `final_config_slab_support_periodic_5beads_tall_rho04_new_1.0_1.0_14000002.data` (on Expanse in
> `~/Documents/lammps_data/input_data/`, copied there by `chain_after_slab.batch`).  If it is not in
> `lammps_data_files_local/` on this Mac the Config cell falls back to the **14000000** snapshot, which is a
> **smoke-test input only** (its piston was visible to solvent and its Nosé–Hoover NPT left T ≈ 1.24); the converter
> prints a loud note and tags the info-log entry when that happens.


In [ ]:
import sys, importlib
from pathlib import Path
from IPython.display import Image, display
import slab_two_pistons as s2p
s2p = importlib.reload(s2p)          # pick up edits to slab_two_pistons.py without a kernel restart
print('converter code:', Path(s2p.__file__).resolve())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIG -- the only cell to edit
# ══════════════════════════════════════════════════════════════════════════
input_file = s2p.DEFAULT_INPUT            # ..._14000002.data  (production input; pull from Expanse)
if not Path(input_file).exists():
    print('NOTE: 14000002 input not found locally -> using the 14000000 snapshot as a SMOKE-TEST input only')
    input_file = s2p.SMOKE_INPUT

cfg = s2p.Config(
    input_file         = input_file,
    output_file        = None,      # None -> <input stem>_two_pist.data next to the input
    permeate_thickness = 10.0,      # sigma of solvent between support and permeate piston (padded up from the existing ~4-6)
    feed_thickness     = None,      # None = keep the existing top-reservoir solvent as is (~16-25 sigma); a number trims/pads
    margin_perm        = None,      # vacuum below the permeate piston; None -> feed_thickness + 5 (piston can back out by the feed volume)
    margin_feed        = 15.0,      # vacuum above the feed piston (compression expels solvent into the feed -> piston rises;
                                    #   15 sigma covers strains up to ~0.10 on a 135-sigma gel: raise it for deeper sweeps)
    piston_clearance   = 1.0,       # each wet piston plane this far outside the outermost solvent bead
    dry_piston_frac    = 0.5,       # dry piston z as a fraction of the feed reservoir height (from the gel top)
    sheet_source       = 'support', # 'support': copy the input support's (x,y) pattern -> identical bead count (23,316)
                                    # 'hex': fresh hex sheet at sheet_spacing (make_sheet() logic of slab_with_support_periodic.ipynb)
    sheet_spacing      = 0.2,       # only for sheet_source='hex'; snapped to tile lx, ly exactly
    seed               = 42,
    log_info           = True,      # append the entry to ../slab_data_file_info.md
)

## Convert

Steps (see the module docstring): parse (isolate_gel parser + Velocities) → delete the old piston → measure the gel
z-extent (BB / percentile / Rg) → unwrap z so the box reads *permeate | support | gel | feed* → measure the bulk
solvent density in the feed reservoir → pad the permeate reservoir (only its *empty* z-intervals, to bulk density,
0.8 σ overlap rejection) → build the three sheets → shift so `zlo = 0`, wrap x,y, renumber → **validate** → write.


In [ ]:
info = s2p.convert(cfg)

## Validation

`convert` refuses to write unless every check passes; the cells below re-run the checks on the written file
(the same routine the CLI `--self-check-only` uses) and confirm `lx`, `ly` and the gel coordinates are identical
to the input up to the uniform z shift.


In [ ]:
val = s2p.self_check(cfg.output_file)
ok, msg = s2p.check_gel_unchanged(cfg.input_file, cfg.output_file)
print(('gel-unchanged check: OK -- ' if ok else 'gel-unchanged check: FAILED -- ') + msg)
assert val['ok'] and ok

In [ ]:
s2p.print_layout(info)
display(Image(filename=info['png']))

## Info-log entry

Printed by `convert` and (with `log_info=True`) already appended to `slab_data_file_info.md`.  Copy the data file to
`~/Documents/lammps_data/input_data/` on Expanse (name = `DATANAME` in the `*_two_pist.batch` files).


In [ ]:
print(s2p.info_log_entry(info))
print('DATANAME for the batch files:', Path(cfg.output_file).stem)